<a href="https://colab.research.google.com/github/yilinw762/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026_09_23_%E2%80%94_Cleaning_Clinic_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [5]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [6]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [7]:
print('shape:', df.shape)
print('dtypes:\n', df.dtypes)
print('null counts:\n', df.isna().sum())
print('exact duplicates:', df.duplicated().sum())

shape: (8, 6)
dtypes:
 order_id      int64
item            str
category        str
qty         float64
price           str
ts              str
dtype: object
null counts:
 order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
exact duplicates: 1


**Problems found:**

1. Order 1 appears twice as an exact duplicate.
2. Prices mix dollar-prefixed strings and bare numbers.
3. Order 3 has no quantity.
4. Order 5 has a negative quantity, which may represent a refund.
5. Categories vary in case and punctuation (Food/food, RainGear/rain-gear).
6. Item names vary in case, whitespace, and spelling (Cheeseburger/cheese burger).
7. Order 7 has no item name.
8. Timestamps use mixed formats, and order 6 has no timestamp.

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [8]:
removed = int(df.duplicated().sum())
clean = df.drop_duplicates().copy()
log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [9]:
price_before = clean['price'].copy()
clean['price'] = pd.to_numeric(clean['price'].str.strip().str.replace('$', '', regex=False)).astype(float)
assert clean['price'].dtype == float
log('price', 'converted text prices to numeric dollars', len(price_before))

[price] converted text prices to numeric dollars (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [10]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')
missing = int(clean['qty'].isna().sum())
negative = int(clean['qty'].lt(0).sum())
clean = clean.loc[clean['qty'].notna()].copy()
log('missing quantity', 'excluded unknown quantities; no evidence supports imputing units', missing)
log('negative quantity', 'retained negative quantities as refunds for net revenue', negative)

[missing quantity] excluded unknown quantities; no evidence supports imputing units (1 row(s))
[negative quantity] retained negative quantities as refunds for net revenue (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [11]:
print('before:', sorted(clean['category'].unique()))
category_before = clean['category'].copy()
normalized = category_before.str.lower().str.strip().str.replace(r'[^a-z0-9]', '', regex=True)
CATEGORY_MAP = {'food': 'food', 'merch': 'merchandise', 'apparel': 'apparel', 'raingear': 'rain gear'}
clean['category'] = normalized.map(CATEGORY_MAP)
assert clean['category'].notna().all(), 'Unmapped category'
print('after:', sorted(clean['category'].unique()))
log('categories', f'normalized categories: {category_before.nunique()} distinct before, {clean["category"].nunique()} after', int(category_before.ne(clean['category']).sum()))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after: ['apparel', 'food', 'merchandise', 'rain gear']
[categories] normalized categories: 6 distinct before, 4 after (5 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [12]:
item_before = clean['item'].copy()
normalized = item_before.str.lower().str.strip().str.replace(r'[^a-z0-9 ]', '', regex=True).str.replace(r'\s+', ' ', regex=True)
ITEM_MAP = {'cheese burger': 'cheeseburger'}
clean['item'] = normalized.replace(ITEM_MAP)
missing_items = int(clean['item'].isna().sum())
clean['item'] = clean['item'].fillna('unknown item')
log('item names', 'normalized case, spacing, punctuation, and cheese burger spelling', int(item_before.notna().mul(item_before.ne(clean['item']).fillna(False)).sum()))
log('missing item', 'labeled unknown item; retained known quantity and price for revenue', missing_items)

[item names] normalized case, spacing, punctuation, and cheese burger spelling (4 row(s))
[missing item] labeled unknown item; retained known quantity and price for revenue (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [13]:
ts_before = clean['ts'].copy()
clean['ts'] = pd.to_datetime(ts_before, format='mixed', errors='coerce')
failed = int((ts_before.notna() & clean['ts'].isna()).sum())
missing_ts = int(ts_before.isna().sum())
print('parse failures:', failed, '| missing timestamps:', missing_ts)
clean['hour'] = clean['ts'].dt.hour.astype('Int64')
log('timestamps', 'parsed mixed formats; interpreted slash dates as month/day/year', int(clean['ts'].notna().sum()))
log('missing timestamps', 'retained rows with NaT and missing hour; revenue is still known', missing_ts + failed)

parse failures: 0 | missing timestamps: 1
[timestamps] parsed mixed formats; interpreted slash dates as month/day/year (5 row(s))
[missing timestamps] retained rows with NaT and missing hour; revenue is still known (1 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [14]:
assert not clean.duplicated().any()
assert clean['order_id'].is_unique
assert len(clean) == raw_rows - removed - missing
assert clean['price'].dtype == float and clean['price'].ge(0).all()
assert clean['qty'].notna().all()
assert int(clean['qty'].lt(0).sum()) == negative
assert set(clean['category']) == {'food', 'merchandise', 'apparel', 'rain gear'}
assert clean['item'].notna().all()
assert clean.loc[clean['order_id'].eq(2), 'item'].iloc[0] == 'cheeseburger'
assert pd.api.types.is_datetime64_any_dtype(clean['ts'])
assert clean.loc[clean['order_id'].eq(2), 'ts'].iloc[0] == pd.Timestamp('2026-09-05 12:40')
assert clean['hour'].isna().equals(clean['ts'].isna())
clean['revenue'] = clean['qty'] * clean['price']
assert np.isclose(clean['revenue'].sum(), 88.5)
print('rows:', len(clean))
print('net units:', clean['qty'].sum())
print('net revenue:', round(clean['revenue'].sum(), 2))
print('distinct categories:', clean['category'].nunique())

rows: 6
net units: 7.0
net revenue: 88.5
distinct categories: 4


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [15]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,converted text prices to numeric dollars,7
2,missing quantity,excluded unknown quantities; no evidence suppo...,1
3,negative quantity,retained negative quantities as refunds for ne...,1
4,categories,"normalized categories: 6 distinct before, 4 after",5
5,item names,"normalized case, spacing, punctuation, and che...",4
6,missing item,labeled unknown item; retained known quantity ...,1
7,timestamps,parsed mixed formats; interpreted slash dates ...,5
8,missing timestamps,retained rows with NaT and missing hour; reven...,1


**The decision that mattered most:** Retaining the sale with a missing timestamp preserves $24 of known revenue. Its quantity and price are present, so it belongs in the overall total even though it cannot contribute to hourly reporting.

**Revenue with it:** $88.50. **Revenue without it:** $64.50.

For comparison, dropping the refund would increase revenue by $18 to $106.50; retaining the duplicate would add $15; imputing one unit for the missing quantity would add $12; dropping the unknown item would subtract $12. The refund interpretation is an assumption to confirm with the data owner. Missing quantities are excluded because their revenue cannot be determined from the available data.

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [16]:
# Checkpoint: compare the largest row-inclusion decision.
rows_after = len(clean)
revenue_after = round(float(clean['revenue'].sum()), 2)
biggest_decision = 'Retained known sales with a missing timestamp; dropping them would remove $24.'
revenue_other_way = round(float(clean.loc[clean['ts'].notna(), 'revenue'].sum()), 2)
print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 6
revenue: 88.5
decision that mattered: Retained known sales with a missing timestamp; dropping them would remove $24.
revenue the other way: 64.5
